In [0]:
from pyspark.sql.functions import *

fact = spark.table("bankingdata.gold.fact_transaction")
dim_card = spark.table("bankingdata.gold.dim_card")

### KPI Data

1. Total Transaction Amount

In [0]:
kpi_total_amount = fact.agg(sum("amount").alias("total_amount"))

2. Total Transactions Count

In [0]:
kpi_total_txn = fact.agg(count("*").alias("total_transactions"))

3. High Value Transactions

In [0]:
kpi_high_value = fact.filter(col("is_high_value") == True) \
                     .agg(count("*").alias("high_value_txn"))

4. Revenue per Branch

In [0]:
kpi_branch_revenue = fact.groupBy("branch_id") \
                        .agg(sum("amount").alias("total_revenue"))

5. Top Customers

In [0]:
kpi_top_customers = fact.groupBy("customer_id") \
                       .agg(sum("amount").alias("total_spend")) \
                       .orderBy(desc("total_spend")) \
                       .limit(5)

6. Transactions by Card Type

In [0]:
kpi_card_type = fact.join(dim_card, fact.card_id == dim_card.CardID, "inner") \
                    .groupBy("CardType") \
                    .agg(count("*").alias("txn_count"))

7. Average Transaction Amount

In [0]:
kpi_avg_amount = fact.agg(avg("amount").alias("avg_txn_amount"))

### Graph Datasets

1. Transaction Trend

In [0]:
graph_trend = fact.withColumn("txn_date", to_date("event_time")) \
                  .groupBy("txn_date") \
                  .agg(sum("amount").alias("total_amount")) \
                  .orderBy("txn_date")

2. Revenue by Branch

In [0]:
graph_branch = fact.groupBy("branch_id") \
                   .agg(sum("amount").alias("total_amount"))

3. Card Type Distribution

In [0]:
graph_card = fact.join(dim_card, fact.card_id == dim_card.CardID) \
                 .groupBy("CardType") \
                 .agg(count("*").alias("txn_count"))

4. Merchant Category

In [0]:
graph_merchant = fact.groupBy("merchant_category") \
                     .agg(count("*").alias("txn_count"))

5. Top Customers

In [0]:
graph_top_customers = fact.groupBy("customer_id") \
                         .agg(sum("amount").alias("total_spend")) \
                         .orderBy(desc("total_spend")) \
                         .limit(10)

6. High Value vs Normal

In [0]:
graph_high_value = fact.groupBy("is_high_value") \
                       .agg(count("*").alias("txn_count"))